<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;
  "></div>
  <b>02. Bayesian Mixture Models with MCMC</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Going Fully Bayesian with Mixtures](#-part-i-going-fully-bayesian)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Dirichlet-Multinomial Mixture Model](#-part-iii-dirichlet-multinomial)**
    - 3.1 [The Dirichlet Prior](#the-dirichlet-prior)
    - 3.2 [Full Bayesian GMM Specification](#full-bayesian-gmm)
- **4. [MCMC Inference for Bayesian GMMs](#-part-iv-mcmc-inference)**
- **5. [Posterior Analysis & Visualization](#-part-v-posterior-analysis)**
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Going Fully Bayesian with Mixtures</span>

Instead of point estimates (EM/MLE), the **fully Bayesian** approach places priors on **all parameters** and infers their full posterior distributions.

| **Aspect** | **MLE/EM** | **Fully Bayesian** |
|-----------|-----------|-------------------|
| Weights | Point estimate | Posterior distribution via Dirichlet prior |
| Means | Point estimate | Posterior distribution via Normal prior |
| Covariances | Point estimate | Posterior distribution via Inverse-Wishart prior |
| Number of components | Must be specified | Can be inferred (Dirichlet Process) |
| Uncertainty | No | Full posterior uncertainty |
| Overfitting risk | Higher | Lower (automatic Occam's razor) |

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

tfd = tfp.distributions
tfb = tfp.bijectors

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Dirichlet-Multinomial Mixture Model</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. The Dirichlet Prior</span>

The **Dirichlet distribution** is the natural prior for mixing weights because it outputs probability vectors:

$$\pi \sim \text{Dirichlet}(\alpha_1, \ldots, \alpha_K)$$

- $\alpha_k = 1$ for all $k$: **Uniform** over the simplex
- $\alpha_k < 1$: Sparse — favors few active components
- $\alpha_k > 1$: Concentrated — favors more uniform weights

In [ ]:
# Visualize Dirichlet distributions with different concentrations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, alpha, title in zip(axes, 
    [[0.1, 0.1, 0.1], [1.0, 1.0, 1.0], [10.0, 10.0, 10.0]],
    ['Sparse (α=0.1)', 'Uniform (α=1.0)', 'Concentrated (α=10.0)']
):
    dirichlet = tfd.Dirichlet(concentration=alpha)
    samples = dirichlet.sample(1000).numpy()
    
    # Plot on a ternary-like scatter
    ax.scatter(samples[:, 0], samples[:, 1], c=samples[:, 2], 
               cmap='viridis', alpha=0.3, s=10)
    ax.set_xlabel('$\\pi_1$', fontsize=13)
    ax.set_ylabel('$\\pi_2$', fontsize=13)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Dirichlet Prior: Effect of Concentration Parameter',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Full Bayesian GMM Specification</span>

The complete generative model:

$$\pi \sim \text{Dirichlet}(\alpha), \quad \mu_k \sim \mathcal{N}(0, \sigma_0^2 I), \quad z_n \sim \text{Cat}(\pi), \quad x_n \sim \mathcal{N}(\mu_{z_n}, \Sigma_{z_n})$$

In [ ]:
# Generate synthetic data
np.random.seed(42)
true_data = np.vstack([
    np.random.multivariate_normal([-3, -2], [[0.5, 0], [0, 0.5]], 150),
    np.random.multivariate_normal([2, 3], [[0.8, 0], [0, 0.6]], 200),
    np.random.multivariate_normal([5, -1], [[0.4, 0], [0, 0.7]], 100),
]).astype(np.float32)
np.random.shuffle(true_data)

N, D = true_data.shape
K = 3  # Number of components

print(f"Data: {N} points in {D}D")
print(f"Components: K={K}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. MCMC Inference for Bayesian GMMs</span>

In [ ]:
# Define the log posterior for the Bayesian GMM
# We marginalize out z (cluster assignments) analytically
data_tf = tf.constant(true_data)

def bayesian_gmm_log_prob(log_weights_raw, means, log_scales):
    """
    Log posterior for a Bayesian GMM.
    log_weights_raw: unconstrained weights (K,)
    means: component means (K, D)
    log_scales: log of component scales (K, D)
    """
    # Transform to constrained space
    weights = tf.nn.softmax(log_weights_raw)
    scales = tf.nn.softplus(log_scales) + 1e-5
    
    # Priors
    lp = tfd.Dirichlet(tf.ones(K)).log_prob(weights)
    lp += tf.reduce_sum(tfd.Normal(0., 10.).log_prob(means))
    lp += tf.reduce_sum(tfd.HalfNormal(5.).log_prob(scales))
    
    # Likelihood (marginalized over z)
    gmm = tfd.MixtureSameFamily(
        mixture_distribution=tfd.Categorical(probs=weights),
        components_distribution=tfd.MultivariateNormalDiag(
            loc=means, scale_diag=scales
        )
    )
    lp += tf.reduce_sum(gmm.log_prob(data_tf))
    
    return lp

# MCMC sampling
kernel = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=bayesian_gmm_log_prob,
        step_size=0.01
    ),
    num_adaptation_steps=500,
    target_accept_prob=0.75
)

@tf.function
def sample_bayesian_gmm():
    return tfp.mcmc.sample_chain(
        num_results=1500,
        num_burnin_steps=1000,
        current_state=[
            tf.zeros(K),                          # log_weights_raw
            tf.random.normal([K, D]) * 3,          # means
            tf.zeros([K, D]),                       # log_scales
        ],
        kernel=kernel,
        trace_fn=None
    )

post_weights_raw, post_means, post_log_scales = sample_bayesian_gmm()
post_weights = tf.nn.softmax(post_weights_raw, axis=-1)
post_scales = tf.nn.softplus(post_log_scales) + 1e-5

print(f"Posterior samples shape:")
print(f"  Weights: {post_weights.shape}")
print(f"  Means:   {post_means.shape}")
print(f"  Scales:  {post_scales.shape}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Posterior Analysis & Visualization</span>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Posterior weights
for k in range(K):
    axes[0].hist(post_weights[:, k].numpy(), bins=40, alpha=0.6,
                 label=f'Component {k+1}', density=True)
axes[0].set_xlabel('Weight', fontsize=13)
axes[0].set_ylabel('Density', fontsize=13)
axes[0].set_title('Posterior: Mixing Weights', fontsize=15, fontweight='bold')
axes[0].legend()

# Posterior means (x-coordinate)
for k in range(K):
    axes[1].hist(post_means[:, k, 0].numpy(), bins=40, alpha=0.6,
                 label=f'Component {k+1}', density=True)
axes[1].set_xlabel('Mean ($x_1$ coordinate)', fontsize=13)
axes[1].set_title('Posterior: Component Means', fontsize=15, fontweight='bold')
axes[1].legend()

# Posterior predictive density
xx, yy = np.meshgrid(np.linspace(-6, 8, 100), np.linspace(-5, 6, 100))
grid_points = np.stack([xx.ravel(), yy.ravel()], axis=-1).astype(np.float32)

# Average over posterior samples
n_post = 50
density_avg = np.zeros(len(grid_points))
for i in range(n_post):
    idx = np.random.randint(len(post_weights))
    gmm_i = tfd.MixtureSameFamily(
        mixture_distribution=tfd.Categorical(probs=post_weights[idx]),
        components_distribution=tfd.MultivariateNormalDiag(
            loc=post_means[idx], scale_diag=post_scales[idx]
        )
    )
    density_avg += gmm_i.prob(grid_points).numpy()
density_avg /= n_post

axes[2].contourf(xx, yy, density_avg.reshape(xx.shape), levels=30, cmap='viridis')
axes[2].scatter(true_data[:, 0], true_data[:, 1], c='white', s=5, alpha=0.3)
axes[2].set_xlabel('$x_1$', fontsize=13)
axes[2].set_ylabel('$x_2$', fontsize=13)
axes[2].set_title('Bayesian Posterior Predictive Density', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| Dirichlet prior | Natural conjugate prior for mixture weights |
| Bayesian GMM | Full posterior over all parameters, not just point estimates |
| MCMC inference | Exact posterior samples (asymptotically) |
| Non-centered parameterization | Often needed for efficient HMC |
| Posterior predictive | Average predictions over parameter uncertainty |
| Label switching | Common issue in Bayesian mixtures — be aware! |